# Measurement · validity  `[EVAL]`

**Is the ruler trustworthy?** The `arms/*`, `lookahead/*` and `method/*` families ask what the arms
did; this one asks whether the instrument that says so can be believed. Three questions, all read
from the `data/eval_scores/` lake that the paid `notebooks/scoring/Judge_Reliability.ipynb` writes
into — this notebook only *reads*, so it costs nothing and renders inside `render_results.py`.

- **§1 · Judge reliability** — the oracle's own repeatability (ICC), plus a decoupled second judge:
  do the endpoint contrasts keep their sign under a grader from a different model family that never
  played the patient?
- **§2 · Multi-judge** — where the variance in an arm mean actually comes from, whether gains
  transfer to a held-out grader, and at what effect size the two judges start agreeing.
- **§3 · Judge saturation** — the one place the two graders stop agreeing *per conversation*, and
  the test that separates "the conversations became indistinguishable" from "the trained-against
  grader ran out of range". A re-cut of §1's and §2's frames; no extra load.

> **Judge-invariant — which is why it is its own family.** Every artifact here contains *both*
> graders, so `reliability.py` loads them explicitly and ignores `EDA_JUDGE`. Exports go to
> `results/measurement/validity/{figures,tables}/` with **no `<judge>/` level**: a path naming one
> grader would assert that grader produced a cross-judge figure. `render_results.py` renders this
> notebook exactly once.
>
> **Every judged model state on one axis** (2026-08-18 reorg). The retired `L0`/`L5` views split the
> judged grid by K, which left the K=0 endpoint contrasts with `primary_n = 0` in the `L5` copy of
> `second_judge_contrasts` (the primary frame there held only K=5 arms). There is one view now; §1
> asserts every contrast row is paired on the full 96 personas.
>
> Split out of `5_Training_and_Reliability` on 2026-07-29 — that notebook is training-side and
> refuses a second judge, which forced these eval-side, cross-judge artifacts to be written under
> the primary oracle's folder.


In [ ]:
import sys, os
_p = os.path.abspath(".")                      # find eda/ (the dir holding eda_analysis/) from any depth
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "eda_analysis")):
    _p = os.path.dirname(_p)
sys.path.insert(0, _p)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 185, "display.max_columns", 50)

import os, eda_analysis
from eda_analysis import exports, plotting
from eda_analysis import reliability as rel
cfg = eda_analysis.EdaConfig(family="measurement/validity", judge=os.environ.get("EDA_JUDGE", ""))
S = eda_analysis.notebook_setup(cfg)
exports.reset_results()   # clears only THIS family's generated figures/tables (never SUMMARY.md)
exports.save_provenance(cfg, S.SCORES)   # re-stamp figures/_provenance.md (reset just removed the one notebook_setup wrote)
# NOTE: no EDA_JUDGE handling, deliberately (a judge-invariant family: notebook_setup already
# ignores it). Every section reads EVERY judge from the score lake via reliability.py, so the
# active-judge knob would change nothing here; S.SCORES is used only to intersect the judged
# model states with the arms the config allows (default = every arm on disk).


## 1 · Judge reliability — oracle ICC + a decoupled second judge  `[EVAL]`

**Purpose.** Answer the two measurement-validity questions the rest of the EDA has to assume. Repeatability is measured on the anchor-model subset (base + endpoints + the GRPO peak × Q1/Q2/MICI × 96 convs); agreement runs on every cell both judges have scored — the full judged-state × 8-rubric grid (the state count is in `multijudge_coverage`; it grows as a run advances). Scored by `Judge_Reliability.ipynb` (paid, run manually); this section only READS `data/eval_scores/`, so it stays free and inside `render_results.py`.

1. **Repeatability (LIMITATIONS §1).** The same oracle re-scoring the same conversations 3×, seeds differing and nothing else → **ICC(2,1)** + mean |Δ|. This is the instrument's own measurement error.
2. **Second judge (LIMITATIONS §2).** The simulated patient and the grading oracle are the same model (`gpt-4o-mini`), so the generator and evaluator are coupled. A different-family judge (Claude Haiku 4.5) that never played the patient breaks that coupling.

**Read.** Agreement is bounded by *both* raters' noise — compare `pearson_r` to the `ceiling` column, never to 1.0. The `bias` column is a LEVEL offset (a harsher judge marks everything down) and is irrelevant to the thesis, whose claims are all *contrasts*: the load-bearing panel is **contrast preservation** — if the PTO−GRPO endpoint gap keeps its sign under a decoupled judge, the headline is not an artifact of the shared patient/oracle model. `second_judge_contrasts` checks the two hand-picked K=0 endpoint pairs (`reliability.DEFAULT_CONTRAST_PAIRS`, paired on `file_index`, valid at matched iterations); every other pair — including the K=5 states — is in §2's persona-paired all-pairs table.


In [ ]:
# AUTO-DISCOVERED from what the second judge has actually scored — not a hardcoded subset.
# This section began life on a 4-model x 3-metric anchor subset; the full sweep covers every
# model states x 8 rubrics, and hardcoding would silently keep reporting the old corner of a grid
# that has since been completed.
if rel.available():
    _cov = rel.coverage_table(rel.load_judge_long(rel.second_judge_tags()[0]))
    JUDGE_METRICS = [m for m in eda_analysis.QUESTIONNAIRE_ORDER if m in set(_cov.metric)]
    JUDGE_MODELS = sorted(set(_cov.model))
else:
    JUDGE_METRICS, JUDGE_MODELS = [], []
_in_scope = [m for m in JUDGE_MODELS if m in set(S.SCORES.model)]
JUDGE_MODELS = _in_scope or JUDGE_MODELS          # respect the config's arm filter (default: every arm)
print(f"[judge] {len(JUDGE_MODELS)} model states x {len(JUDGE_METRICS)} metrics scored by the second judge")

if not rel.available():
    print("No re-scoring data on disk — run Judge_Reliability.ipynb first (writes data/eval_scores/judge=<tag>/rep=<r>/).")
elif not _in_scope:
    print("none of the judged model states is inside the config's arm filter — section skipped.")
else:
    TAG = rel.second_judge_tags()[0]
    JUDGE_NAME = rel.judge_display(TAG)

    # ── 1a · repeatability of the primary oracle ──────────────────────────────
    REP = rel.repeatability()
    if not REP.empty:
        display(rel.repeatability_by_metric(REP))
        exports.save_table(rel.repeatability_by_metric(REP), "oracle_repeatability_by_metric",
                           caption="Oracle repeatability per metric (primary oracle gpt-4o-mini, anchor-model subset): ICC(2,1) across 3 re-scorings of the same conversations (seeds differ only) + the mean per-conversation |delta| between reps. The citable 'oracle noise' figure.")
        exports.save_table(REP, "oracle_repeatability_icc",
                           caption="Oracle repeatability per (metric, model), primary oracle gpt-4o-mini: ICC(2,1) + mean |delta| across 3 re-scorings of the same 96 conversations.")
        fig = plotting.oracle_repeatability_bars(REP, metrics=JUDGE_METRICS)
        if fig:
            exports.save_fig(fig, "oracle_repeatability_icc",
                             caption="ICC(2,1) per model and metric from 3 re-scorings of the same conversations by the primary oracle (gpt-4o-mini); dotted = Koo & Li good (0.75) / excellent (0.90).")
            plt.show()

    # ── 1b · second judge vs the primary oracle ───────────────────────────────
    JL = rel.load_judge_long(TAG, reps=[0])
    PL = rel.load_primary_long(JUDGE_MODELS, JUDGE_METRICS)
    AGR = rel.agreement(JL, PL, REP)
    display(AGR)
    exports.save_table(AGR, "second_judge_agreement",
                       caption=f"Per-conversation agreement between {JUDGE_NAME} (held-out) and the primary oracle (gpt-4o-mini, the training reward), per (metric, model) over all {len(JUDGE_MODELS)} model states: Pearson r, Spearman rho, level bias (judge minus primary), and the attenuation ceiling implied by both judges' ICC.")
    fig = plotting.judge_agreement_scatter(JL, PL, agr_tab=AGR, metrics=JUDGE_METRICS,
                                           judge_name=JUDGE_NAME)
    if fig:
        exports.save_fig(fig, "judge_agreement_scatter",
                         caption=f"Per-conversation scores, {JUDGE_NAME} vs the primary oracle (gpt-4o-mini), one panel per metric, all model states pooled. Distance from the dashed identity line is level bias; scatter around a trend is rank disagreement.")
        plt.show()

    display(rel.arm_means_by_judge(JL, PL, JUDGE_NAME))

    # ── 1c · THE defense check: does the contrast survive the judge swap? ──────
    CON = rel.contrasts(JL, PL, JUDGE_METRICS)
    display(CON)
    # The one-view fix: with every arm in the primary frame, no contrast row may be unpaired.
    # (The retired L5 view rendered these K=0 pairs with primary_n = 0 and same_sign = False.)
    _bad = CON[(CON.primary_n == 0) | (CON.judge_n == 0)]
    assert _bad.empty, f"unpaired contrast rows (primary_n or judge_n == 0):\n{_bad}"
    print(f"[contrasts] {len(CON)} rows, primary_n {int(CON.primary_n.min())}-{int(CON.primary_n.max())}, "
          f"judge_n {int(CON.judge_n.min())}-{int(CON.judge_n.max())} — every row paired")
    exports.save_table(CON, "second_judge_contrasts",
                       caption=f"Contrast preservation: each hand-picked K=0 endpoint contrast (a minus b; MICI lower = better, so a negative delta favours a) as a paired delta over the 96 matched conversations (file_index pairing) under the primary oracle (gpt-4o-mini) and under {JUDGE_NAME}, with same_sign. The defense against the patient=oracle coupling in LIMITATIONS section 2. K=5 states and every other pair: multijudge_all_pairs_contrasts.")
    fig = plotting.judge_contrast_bars(CON, metrics=JUDGE_METRICS, judge_name=JUDGE_NAME)
    if fig:
        exports.save_fig(fig, "judge_contrast_preservation",
                         caption=f"K=0 endpoint contrasts under both judges (primary oracle gpt-4o-mini vs held-out {JUDGE_NAME}). Same-sign bars mean the result does not depend on the grader also having played the patient ({JUDGE_NAME} is a different model family). MICI: lower = better.")
        plt.show()

    print("\nVERDICT:", rel.summary_line(REP, AGR, CON))


## 2 · Multi-judge — variance sources, transfer, and resolution  `[EVAL]`

**Purpose.** §1 asked *does the contrast survive a second judge?* (yes, on two hand-picked pairs). This section asks the three follow-ups a defence actually turns on. Free — reads the same `data/eval_scores/` lake. Everything below runs over **every fully-scored (metric, model) cell** of all four arms — every judged model state the sweep covers, the live count and each arm's scored support being in `multijudge_coverage`.

**2a · Both judges, side by side.** A dumbbell per model state: bar *length* is the level offset, bar *order* is the claim. The two are never averaged — see the note above.

**2b · All pairwise contrasts.** Every pair of the judged states × 8 rubrics — C(N,2) × 8 contrasts, N = the judged-state count the caption derives (`len(CONTRAST_MODELS)`) — under both judges. That is far too many to read as a table (the `.md` is a head excerpt; the leaf workbook holds every row), so the *rate* over it is what the thesis quotes, reported as a **sign-preservation ladder** by claimed effect size rather than a single pooled number. Pairing is on the recovered `persona_id`, not `file_index`: the trainer reshuffles the 96 personas every iteration, so a `file_index` join across unmatched iterations pairs unrelated conversations. Means are unaffected by that, but `dz` and the CI are not — and those are what a thesis table reports.

**2c · Variance decomposition.** Two-way random effects over arms × judges, on the arm means the thesis actually reports. Three components: **arm** (signal), **judge level** (large, and harmless — it cancels in every contrast), and **arm × judge** (the only one that threatens a claim: an ordering that depends on who is grading). `dependability_k1` is the generalizability coefficient for an arm mean read off a single judge — the number to quote when asked how far one judge's ranking can be trusted; `k2` is the same with both judges averaged, which is the honest answer to *"would a second judge help?"*.

**2d · Gain retention — the reward-hacking test.** What fraction of each arm's gain over Base survives the judge swap. Because the primary judge *was* the training reward and the second judge is held out, `Δ(judge) / Δ(primary)` is a **train/test generalization ratio**, not a reliability statistic: ~1.0 means the gain is a real behaviour change both judges see; ~0 means it lived only in the grader that was optimized. Uniform retention across arms is scale compression and uninteresting — the signal is retention that *differs by arm on one metric while staying flat on another*. The reference is ONE shared base draw (`PTOExp3_LA0_Base`, the long-standing convention) for all four arms; per-K and per-method references are the `lookahead/transfer` family's job.

With every iteration scored by both judges, retention is also a **trajectory**: reward hacking is a process, so the sharper question is not *"did this endpoint transfer?"* but ***"at which iteration did the gains stop transferring?"*** — a line declining with training estimates when the policy began fitting its grader, which no single-endpoint comparison can give you. A line that stops before the x-axis does is retention suppressed by the |Δ primary| floor, not a missing model state; each arm's scored support is in `multijudge_coverage`.

**2e · Concordance vs effect size.** *"When the primary judge reports a gap of at least x, how often does the second judge agree on the direction?"* — a curve, not a scalar, because a single r is dominated by the 1.2–1.7 point level offset that cancels in every contrast, while a rank statistic discards the magnitude that decides whether a gap matters. ⚠ **Each point is a pair of single conversations**; the thesis compares 96-conversation means, which resolve ~10× better. Do not read a bin height as confidence in an arm-level claim — that is what 2a and `dependability_k1` are for. Exact primary-judge ties are excluded (they state no ordering to reproduce; counting them as failures pushes the smallest bin below chance).


In [ ]:
if not rel.available() or not _in_scope:
    print("Multi-judge section skipped (no second-judge scores for the arms in scope).")
else:
    # The "gain over what?" baseline for 2d — derived from the judged frame, never hardcoded.
    # Prefer the PTO K=0 base to match the long-standing convention (one shared reference draw);
    # a hardcoded name absent from the frame once made gain_retention() skip every metric and
    # save an EMPTY multijudge_gain_retention.md (caught 2026-08-18).
    _bases = sorted(m for m in set(JL.model) & set(PL.model) if m.endswith("_Base"))
    _bases = [m for m in _bases if m.startswith("PTOExp3")] + _bases
    if not _bases:
        raise SystemExit("no *_Base model in the judged frame — cannot anchor gain retention")
    REFERENCE_MODEL = _bases[0]
    print(f"[retention] reference base = {REFERENCE_MODEL}")
    JUDGE_N_EXPECTED = 96                  # conversations per (metric, model) in a complete sweep

    # A second-judge sweep can land PARTIALLY (rate limits, expired batch, exhausted credit).
    # Partial cells are unbiased but less precise, and persona-paired stats collapse across two
    # partial arms — so restrict every table below to fully-scored cells and say what was dropped.
    COV = rel.coverage_table(JL, n_expected=JUDGE_N_EXPECTED)
    exports.save_table(COV, "multijudge_coverage",
                       caption=f"Conversations scored by {JUDGE_NAME} per (metric, model), out of {JUDGE_N_EXPECTED}, over every model state in the score lake. Section 2 analyses only cells marked complete; partial cells are reported here so a truncated sweep is never mistaken for full coverage.")
    if not COV.complete.all():
        print(f"[coverage] second-judge sweep is INCOMPLETE — "
              f"{int(COV.complete.sum())}/{len(COV)} cells fully scored "
              f"({COV.pct.min():.0f}-{COV.pct.max():.0f}% per cell). "
              f"Section 2 falls back to the complete cells only.")
    JL, PL = rel.filter_complete_cells(JL, PL, n_required=JUDGE_N_EXPECTED)
    JUDGE_METRICS = [m for m in JUDGE_METRICS if m in set(JL.metric)]
    if JL.empty or not JUDGE_METRICS:
        raise SystemExit("no fully-scored second-judge cells — nothing to analyse in section 2")

    # Per-arm scored support for the captions, DERIVED and never asserted: the reliability frames
    # are (metric, model, file_index, value), so read arm/iteration off S.SCORES restricted to the
    # states this section actually analyses. constants.support_note returns "" unless an arm really
    # does stop before the others, so a caption can only report a truncation that is in the data.
    _judged = S.SCORES[S.SCORES.model.isin(set(JL.model))]
    _short = eda_analysis.support_note(_judged, subject=f"no later state scored by {JUDGE_NAME}")
    SUPPORT = f"{_short} " if _short else ""
    print(f"[support] {_short or 'every judged arm reaches the same last iteration — no support note'}")

    # ── 2a · both judges side by side (never averaged) ────────────────────────
    fig = plotting.judge_dumbbell(JL, PL, metrics=JUDGE_METRICS, judge_name=JUDGE_NAME)
    if fig:
        # One judged model state per row per panel (the retired views held 22 / 17): the panel height the plotting
        # module picks is sized for the smaller grid and packs the y-tick labels into an unreadable
        # smear. Stretch the figure vertically and pull the header back down before saving.
        _w, _h = fig.get_size_inches()
        fig.set_size_inches(_w, _h * 2.2)
        if fig.legends:
            fig.legends[0].set_bbox_to_anchor((0.5, 1.02))
        if fig._suptitle is not None:
            fig._suptitle.set_y(1.045)
        fig.tight_layout()
        exports.save_fig(fig, "multijudge_arm_means_dumbbell",
                         caption=f"Arm means per model state (all four arms, every state with complete second-judge coverage) under the primary oracle (gpt-4o-mini) and {JUDGE_NAME}. {SUPPORT}Bar LENGTH is the level offset (large, and it cancels in every contrast); bar ORDER is what the thesis claims. Deliberately not averaged: the primary judge was the training reward and the second is held out, and the offset is model-dependent.")
        plt.show()

    # ── 2b · every pairwise contrast, persona-paired ──────────────────────────
    # Every pair of the judged model states x metric. C(N,2) x 8 rows, N derived: the .md is a head
    # excerpt (exports.MD_MAX_BYTES), the leaf workbook holds every row, and the sign-preservation
    # ladder below is the summary the thesis quotes.
    CONTRAST_MODELS = [m for m in JUDGE_MODELS if m in set(JL.model)]
    PAIRS = rel.all_pairs_contrasts(JL, PL, JUDGE_METRICS, models=CONTRAST_MODELS)
    print(f"[contrasts] {len(CONTRAST_MODELS)} model states -> {len(PAIRS)} contrasts "
          f"({len(set(JL.model))} states scored in total)")
    display(PAIRS[["metric", "contrast", "primary_delta", "judge_delta",
                   "judge_ci_lo", "judge_ci_hi", "judge_dz", "same_sign"]])
    exports.save_table(PAIRS, "multijudge_all_pairs_contrasts",
                       caption=f"Every model-state pair x metric ({len(CONTRAST_MODELS)} states, all four arms) under both judges, paired on the recovered persona (a minus b; MICI lower = better). judge_ci_* is a percentile bootstrap over personas (seed 42). same_sign is the defence: {int(PAIRS.same_sign.sum())}/{len(PAIRS)} contrasts keep their direction under {JUDGE_NAME}, which never played the patient. The .md is a head excerpt; the workbook holds every row.")

    # The rate over that table is what the thesis quotes, and it is only interpretable against an
    # effect size: a pooled "88% agree" reads as weak until you see the disagreements sit entirely
    # in gaps too small to claim. Save the ladder so the narrative docs cite a tracked artifact.
    SIGN = rel.sign_preservation(PAIRS)
    SIGN_BY_METRIC = rel.sign_preservation(PAIRS, by=["metric"])
    display(SIGN)
    exports.save_table(SIGN, "multijudge_sign_preservation",
                       caption=f"Share of the {len(PAIRS)} pairwise model-state x metric contrasts (all four arms) whose direction survives the swap to {JUDGE_NAME}, as a function of the gap the primary judge (gpt-4o-mini) reports. Read the row at the effect size you are claiming; the pooled row is dragged down by contrasts too small to claim in the first place.")
    exports.save_table(SIGN_BY_METRIC, "multijudge_sign_preservation_by_metric",
                       caption="The same ladder per rubric. A rubric that preserves sign less often is one whose arm ordering depends on who is grading - compare against dependability_k1 in multijudge_variance_components, which measures the same weakness from a completely different direction. WARNING: the thresholds are ABSOLUTE, so a row is comparable to other rows of the SAME rubric, never across rubrics - PCT and MICI live on a 0-1 scale and never reach 0.25, while Q1/Q2/WAI-SR/MITI are 1-5 or 1-7. The cross-rubric comparison is the all-contrasts row.")

    # ── 2c · where does arm-mean variance come from? ──────────────────────────
    VC_CONV = rel.variance_components_conversation(JL, PL, JUDGE_METRICS)
    VC_ARM = rel.variance_components_arm(JL, PL, JUDGE_METRICS, conv_components=VC_CONV)
    display(VC_ARM)
    exports.save_table(VC_ARM, "multijudge_variance_components",
                       caption=f"Two-way random-effects decomposition of the arm means the thesis reports (all {len(CONTRAST_MODELS)} judged model states x 2 judges): arm (signal) vs judge level (cancels in contrasts) vs arm x judge (ordering that depends on the grader). dependability_k1/k2 = generalizability of an arm mean read off one judge vs both averaged.")
    exports.save_table(VC_CONV, "multijudge_variance_components_per_conversation",
                       caption="The same decomposition at the per-conversation level, per (metric, model). var_resid here is per-conversation judge disagreement, which is what attenuates cross-judge correlations.")
    fig = plotting.variance_decomposition_bars(VC_ARM, metrics=JUDGE_METRICS)
    if fig:
        exports.save_fig(fig, "multijudge_variance_decomposition",
                         caption="Share of arm-mean variance by source (all judged model states, primary oracle gpt-4o-mini vs the held-out judge). A large judge-level slice is harmless (it cancels in contrasts); the arm x judge slice is the only component that threatens a claim.")
        plt.show()

    # ── 2d · does the improvement transfer to a held-out judge? ───────────────
    RET = rel.gain_retention(JL, PL, REFERENCE_MODEL, JUDGE_METRICS)
    display(RET)
    exports.save_table(RET, "multijudge_gain_retention",
                       caption=f"Fraction of each model state's gain over {REFERENCE_MODEL} (one shared reference draw for all four arms) that survives the swap to {JUDGE_NAME}. The primary judge (gpt-4o-mini) was the training reward and the second judge is held out, so this is a train/test generalization ratio: ~1.0 = a real behaviour change; ~0 = a gain that existed only in the optimized grader. Direction-agnostic (MICI lower = better flips both deltas). Persona-paired; CI is a persona bootstrap (seed 42); retention suppressed where |delta_primary| < 0.15.")
    fig = plotting.gain_retention_bars(RET, metrics=JUDGE_METRICS, judge_name=JUDGE_NAME)
    if fig:
        exports.save_fig(fig, "multijudge_gain_retention",
                         caption=f"Gain retention under a held-out judge ({JUDGE_NAME}), all four arms' model states, with persona-bootstrap CIs. Uniform bars across arms = scale compression; one arm collapsing while others hold = that arm's gain did not transfer.")
        plt.show()

    # Retention as a TRAJECTORY: with every iteration scored by both judges, "when did the gains
    # stop transferring?" becomes answerable, which no single-endpoint comparison can do.
    fig = plotting.retention_trajectory(RET, metrics=JUDGE_METRICS, judge_name=JUDGE_NAME,
                                        palette=S.PALETTE)
    if fig:
        exports.save_fig(fig, "multijudge_retention_trajectory",
                         caption=f"Gain retention vs iteration under {JUDGE_NAME}, one line per arm, reference {REFERENCE_MODEL}. {SUPPORT}A line near 1.0 = gains a held-out judge also sees; a line declining with training = a policy progressively fitting the grader it was trained against, with the turn point estimating when that set in. A line that ends or gaps before the axis does is retention suppressed where |delta_primary| < 0.15, not a missing model state - each arm's scored support is in multijudge_coverage.")
        plt.show()

    # Q1-only single panel at column width. The retention claim in the reward-hacking draft rests
    # on the Q1 panel, and the full multi-metric grid is illegible at \columnwidth — this is the
    # figure the paper actually embeds (requested by the draft's Figure-3 legibility TODO).
    if "Q1" in set(RET.metric):
        fig = plotting.retention_trajectory(RET, metrics=["Q1"], ncols=1, judge_name=JUDGE_NAME,
                                            palette=S.PALETTE)
        if fig:
            exports.save_fig(fig, "multijudge_retention_trajectory_Q1",
                             caption=f"Gain retention vs iteration under {JUDGE_NAME}, Q1 only, all four arms — the single panel the retention claim rests on, sized for a one-column figure. Same data as the Q1 panel of multijudge_retention_trajectory.")
            plt.show()

    # ── 2e · how much resolution does a gap of a given size carry? ────────────
    CONC = pd.concat([rel.concordance_by_effect_size(JL, PL, m, scope=s)
                      for m in JUDGE_METRICS for s in ("cross_model", "within_model")],
                     ignore_index=True)
    if not CONC.empty:
        display(CONC.pivot_table(index="bin", columns=["metric", "scope"], values="concordance"))
        exports.save_table(CONC, "multijudge_concordance_by_effect_size",
                           caption=f"P({JUDGE_NAME} agrees on the direction) as a function of the gap the primary judge (gpt-4o-mini) reports, per conversation PAIR (seeded sample of at most 400,000 pairs over all judged model states). Exact primary-judge ties excluded. Not a confidence in any arm-level claim - arm means over 96 conversations resolve far better; see multijudge_all_pairs_contrasts.")
        fig = plotting.concordance_curve(CONC, judge_name=JUDGE_NAME)
        if fig:
            exports.save_fig(fig, "multijudge_concordance_curve",
                             caption=f"Cross-judge ordering agreement ({JUDGE_NAME} vs the primary oracle) vs effect size, per conversation pair. Shows how much per-conversation resolving power a given gap carries - i.e. why 96 conversations per arm are needed.")
            plt.show()

    print("\nVERDICT:", rel.multi_judge_summary_line(VC_ARM, RET, PAIRS))

    # ── ledger · the handful of citable scalars, keyed for the papers ─────────
    # reliability.py ships no *_numbers() builder, so the ledger is assembled here from the tables
    # just saved (every value names its table). Point estimates only — no bootstrap CI keys.
    NUM = {}
    if not REP.empty:
        for r in rel.repeatability_by_metric(REP).itertuples():
            NUM[f"oracle_icc.{r.metric}.icc_2_1"] = {"value": float(r.icc_2_1), "source": "tables/oracle_repeatability_by_metric.md",
                                                     "note": "primary oracle ICC(2,1), mean over anchor models"}
            NUM[f"oracle_icc.{r.metric}.mean_abs_diff"] = {"value": float(r.mean_abs_diff), "source": "tables/oracle_repeatability_by_metric.md",
                                                           "note": "mean per-conversation |delta| between re-scorings"}
    for r in CON.itertuples():
        key = f"contrast.{r.contrast.replace(' − ', '_minus_').replace(' ', '')}.{r.metric}"
        NUM[f"{key}.primary_delta"] = {"value": float(r.primary_delta), "source": "tables/second_judge_contrasts.md", "note": "gpt-4o-mini, file_index-paired, n=%d" % r.primary_n}
        NUM[f"{key}.judge_delta"] = {"value": float(r.judge_delta), "source": "tables/second_judge_contrasts.md", "note": f"{JUDGE_NAME}, file_index-paired, n=%d" % r.judge_n}
        NUM[f"{key}.same_sign"] = {"value": bool(r.same_sign), "source": "tables/second_judge_contrasts.md", "note": "sign preserved under the held-out judge"}
    NUM["contrast.n_same_sign"] = {"value": int(CON.same_sign.sum()), "source": "tables/second_judge_contrasts.md", "note": f"of {len(CON)} hand-picked K=0 endpoint contrasts"}
    NUM["contrast.n_total"] = {"value": int(len(CON)), "source": "tables/second_judge_contrasts.md", "note": "rows in second_judge_contrasts"}
    for r in SIGN.itertuples():
        sub = r.subset.replace("|Δ primary| ≥ ", "abs_primary_ge_").replace(" ", "_").replace(".", "p")
        NUM[f"sign_preservation.{sub}.pct_same_sign"] = {"value": float(r.pct_same_sign), "source": "tables/multijudge_sign_preservation.md",
                                                          "note": f"{int(r.n_same_sign)}/{int(r.n_contrasts)} persona-paired contrasts, all four arms"}
    for r in VC_ARM.itertuples():
        NUM[f"variance.{r.metric}.pct_arm_x_judge"] = {"value": float(r.pct_arm_x_judge), "source": "tables/multijudge_variance_components.md", "note": "share of arm-mean variance in the arm x judge interaction"}
        NUM[f"variance.{r.metric}.dependability_k1"] = {"value": float(r.dependability_k1), "source": "tables/multijudge_variance_components.md", "note": "generalizability of an arm mean read off ONE judge"}
    NUM["multijudge.n_states"] = {"value": int(len(CONTRAST_MODELS)), "source": "tables/multijudge_coverage.md", "note": "model states with complete second-judge coverage"}
    NUM["multijudge.n_pairs_contrasts"] = {"value": int(len(PAIRS)), "source": "tables/multijudge_all_pairs_contrasts.md", "note": "C(n_states,2) x n_metrics"}
    NUM["multijudge.reference_model"] = {"value": REFERENCE_MODEL, "source": "tables/multijudge_gain_retention.md", "note": "gain-retention reference draw"}
    exports.save_numbers("validity_numbers", NUM,
                         caption="Citable scalars of this family (oracle ICC per metric, the hand-picked K=0 endpoint contrasts under both judges, the sign-preservation ladder, per-metric arm x judge share + dependability_k1); every value names the table it was read from.")
    print(f"[ledger] validity_numbers: {len(NUM)} keys")


## 3 · Judge saturation — where a grader stops being able to tell conversations apart  `[EVAL]`

**Purpose.** §1 reports cross-judge agreement as a table with one row per (metric, model state), and §2 pools every state into rates and variance components. Both are the right shape for a *defence* — and both hide the single most consequential measurement fact in this experiment, because it lives in a handful of rows near the end of one arm. Plotted as a trajectory instead of read as a table, **Q1** agreement between the two graders wanders in a band for every arm until GRPO's K=5 late iterations, where it falls off the bottom of the whole judged-state distribution. This section is a **re-cut of artifacts already on disk** (`AGR` from §1, the per-conversation frames from §2) — no new load, no API call.

**What the panels show.**
- **(a) The symptom.** Per-conversation Pearson `r` between the held-out judge and the primary oracle on **Q1**, by iteration, one line per arm, against the median over every judged model state. One arm leaves that band decisively. Note also that **both K=5 arms** decline over their last two iterations while neither K=0 arm does — with two arms per optimizer this cannot separate "a property of this checkpoint" from "a property of long K=5 training", and the second reading is the more interesting one.
- **(b) The mechanism.** Q1 **standard deviation** by iteration for the collapsing arm, under *each* grader separately, on the *same* conversations. The training grader's spread collapses (monotonically) while the held-out grader's stays flat — so the correlation is not falling because the conversations became identical (that would shrink *both* SDs), it is falling because **one ruler ran out of range**: the optimized grader bunches conversations the held-out judge still spreads apart. One-sided saturation of the grader that *was* the training reward is a different failure from homogenised policy output, and the two SDs only **one** of them moving is what separates them.
- **(c) The scope.** Agreement at the collapsing arm's endpoint across all eight rubrics, each read against **its own** judged-state distribution (band = 10th–90th percentile, tick = median), because baseline agreement differs enormously by rubric (PCT ≈0.95, MICI ≈0.52) and an absolute cross-rubric comparison would be meaningless. Read the *size* of the shortfall, not the rank. By size the order at this state is **MITI −0.325, Q1 −0.311, MICI −0.231, Q2 −0.194**, and then a clear gap to the four high-agreement rubrics at only −0.02 to −0.05 (CSQ-8, PCT, MI-SAT, WAI-SR). So the two halves of the training reward rank **2nd and 4th, not 1st and 2nd** — MITI's shortfall is the largest of all eight and MICI's exceeds Q2's. The split that holds is *rewarded **and** behaviour-coding rubrics degrade, global-impression rubrics do not*. ⚠ But MITI and MICI are low-agreement rubrics in *every* arm (medians 0.658 / 0.519), so their values here cannot separate a state-specific collapse from their own baseline noise — they widen the pattern beyond the reward, and they are not independent evidence for it.

**The traps.**
- ⚠ **This is a PER-CONVERSATION statistic and it licenses no arm-level claim, nor the reverse.** §2's sign-preservation ladder is arm-level and stays high; a 96-conversation mean averages away most of the per-conversation disagreement measured here. The two coexist. Do not read panel (a) as "the arm-level results for these states are wrong", and do not read the sign-preservation rate as "so per-conversation agreement at these states is fine".
- ⚠ **These `r` values have no measured ceiling.** Repeatability replicates were only ever bought on the anchor subset, which contains **no K=5 state at all** — so `AGR.ceiling` is `NaN` for every row of panel (a)'s collapsing line, and the drop cannot be benchmarked against an attenuation bound the way §1 otherwise instructs. It is compared to the judged-state *median* instead, which is a distributional reference, not a reliability one. The cell asserts that NaN rather than assuming it.
- ⚠ **Panel (a) is a correlation and panel (b) is a spread — neither is a score.** Levels are never compared across graders anywhere in this notebook (the offset is 1.2–1.7 points and model-dependent). Panel (b) puts both graders on one y-axis *only* because SD is a dispersion in each grader's own units and the claim is about the two *directions of change*, not about which number is larger.
- The collapsing **arm** and the endpoint state are **derived** (whichever judged state agrees worst on Q1, and the arm that owns it), so this section reports the outlier that is in the data rather than one a previous session expected. Q1 itself is a *design* choice, not a search result: it is half the training reward, which is what makes "the optimized grader saturated" a coherent claim at all.


In [ ]:
# A RE-CUT of frames §1 and §2 already built: AGR (agreement per metric x model state) and the
# per-conversation frames JL / PL. Nothing is loaded from disk here and nothing is scored.
if not rel.available() or not _in_scope or "Q1" not in set(AGR.metric):
    print("Judge-saturation section skipped (no Q1 cross-judge agreement on disk).")
else:
    from matplotlib.lines import Line2D

    SAT_METRIC = "Q1"          # DESIGN choice, not a search result: half the training reward, which
                               # is what makes "the optimized grader saturated" a coherent claim.
    PRIMARY_NAME = rel.judge_display(rel.PRIMARY_TAG)

    # arm + iteration + K per model state, DERIVED from S.SCORES rather than parsed out of the
    # model string here: S.SCORES carries the canonical arm labels S.PALETTE is keyed on, so this
    # figure's colours match every other arm figure in the results tree.
    _MAP = S.SCORES[["model", "arm", "iteration", "K"]].drop_duplicates().set_index("model")
    ARM_OF, ITER_OF, K_OF = (_MAP["arm"].to_dict(), _MAP["iteration"].to_dict(), _MAP["K"].to_dict())

    AG = AGR[AGR.model.isin(ARM_OF)].copy()
    AG["arm"] = AG.model.map(ARM_OF)
    AG["iteration"] = AG.model.map(ITER_OF).astype(int)
    AG["K"] = AG.model.map(K_OF).astype(int)

    Q = AG[AG.metric == SAT_METRIC].sort_values(["arm", "iteration"])
    Q_MEDIAN = float(Q.pearson_r.median())                 # panel (a)'s distributional reference
    _worst = Q.nsmallest(2, "pearson_r")
    SAT_ARM = _worst.arm.iloc[0]                           # DERIVED: the arm owning the worst state
    SAT_K = int(_worst.K.iloc[0])
    SAT_LAST = Q[Q.arm == SAT_ARM].sort_values("iteration").iloc[-1]      # its endpoint state
    SAT_COLOUR = S.PALETTE.get(SAT_ARM, "#c0392b")
    print(f"[saturation] {SAT_METRIC}: median r = {Q_MEDIAN:.3f} over {len(Q)} judged states; "
          + "two lowest = " + ", ".join(f"{r.model} {r.pearson_r:.3f}" for r in _worst.itertuples())
          + f" -> collapsing arm = {SAT_ARM} (K={SAT_K})")

    # The "no measured ceiling" caveat is ASSERTED off the table, never assumed: repeatability
    # replicates were only ever bought on the anchor subset, which holds no K=5 state.
    _krows = AG[AG.K == SAT_K]
    _n_ceiling = int(_krows.ceiling.notna().sum()) if "ceiling" in _krows.columns else 0
    CEIL_TXT = (
        f"NO MEASURED CEILING: no K={SAT_K} state carries a repeatability replicate (the anchor "
        f"subset the oracle was re-scored on is K=0 only), so `ceiling` is NaN for all {len(_krows)} "
        f"K={SAT_K} rows and these correlations cannot be benchmarked against a measured attenuation "
        f"ceiling; the judged-state median drawn in (a) is a DISTRIBUTIONAL reference, not a "
        f"reliability one."
        if _n_ceiling == 0 else
        f"CEILING: {_n_ceiling} of the {len(_krows)} K={SAT_K} rows now carry a measured attenuation "
        f"ceiling - read r against that column, not against 1.0.")
    print(f"[saturation] {CEIL_TXT}")

    # ── panel (b) input · SD of the SAME conversations under EACH grader separately ──
    # A spread in each grader's own units. Levels are never compared across graders anywhere here.
    _both = pd.concat([PL[["metric", "model", "file_index", "value"]].assign(judge=PRIMARY_NAME),
                       JL[["metric", "model", "file_index", "value"]].assign(judge=JUDGE_NAME)],
                      ignore_index=True)
    _both = _both[(_both.metric == SAT_METRIC) & (_both.model.map(ARM_OF) == SAT_ARM)]
    SD = _both.groupby(["judge", "model"])["value"].agg(sd="std", n_obs="size").reset_index()
    SD["iteration"] = SD.model.map(ITER_OF).astype(int)
    SD = SD.sort_values(["judge", "iteration"])
    _piv = SD.pivot(index="iteration", columns="judge", values="sd")
    _i0, _iN = int(_piv.index.min()), int(_piv.index.max())
    VAR_RATIO = {j: float(_piv.loc[_iN, j]) ** 2 / float(_piv.loc[_i0, j]) ** 2 for j in _piv.columns}
    # ⚠ VAR_RATIO is a TWO-POINT ratio and is only as stable as its anchor. This figure asserted
    # that the held-out grader's spread "grows 1.410x" until 2026-08-25 -- iteration 0 is that
    # series' MINIMUM, so re-anchoring to iteration 1 gives 1.062x and to the mean of iters 1..N
    # 1.034x. Derive a TREND instead and let the prose follow it, so a moving anchor cannot invert
    # the claim. (The primary's collapse is monotone and survives either way.)
    from scipy.stats import spearmanr as _spearmanr
    SD_TREND = {}
    for _j in _piv.columns:
        _s = _piv[_j].dropna()
        _rho, _p = _spearmanr(_s.index.values, _s.values)
        _alt = {"vs_iter1": float(_piv.loc[_i0 + 1, _j]) if (_i0 + 1) in _piv.index else float("nan"),
                "vs_mean_rest": float(_piv.loc[_piv.index > _i0, _j].mean())}
        SD_TREND[_j] = {"rho": float(_rho), "p": float(_p),
                        "var_ratio_vs_iter1": float(_piv.loc[_iN, _j]) ** 2 / _alt["vs_iter1"] ** 2,
                        "var_ratio_vs_mean_rest": float(_piv.loc[_iN, _j]) ** 2 / _alt["vs_mean_rest"] ** 2,
                        "verdict": ("collapses" if (_rho < 0 and _p < 0.05) else
                                    "grows" if (_rho > 0 and _p < 0.05) else "is flat")}
    for _j, _v in VAR_RATIO.items():
        _t = SD_TREND[_j]
        print(f"[saturation] {SAT_ARM} {SAT_METRIC} SD under {_j}: iter {_i0} {_piv.loc[_i0, _j]:.3f} "
              f"-> iter {_iN} {_piv.loc[_iN, _j]:.3f}  (variance x{_v:.3f}; re-anchored "
              f"x{_t['var_ratio_vs_iter1']:.3f} vs iter {_i0 + 1}, x{_t['var_ratio_vs_mean_rest']:.3f} "
              f"vs mean of the rest) | trend rho={_t['rho']:+.3f} p={_t['p']:.3f} -> {_t['verdict'].upper()}")
    _PRIM_V, _HELD = SD_TREND[PRIMARY_NAME], [j for j in _piv.columns if j != PRIMARY_NAME]
    SD_STORY = (f"the TRAINED-AGAINST grader's spread {_PRIM_V['verdict']}"
                + (f" while the held-out grader's {SD_TREND[_HELD[0]]['verdict']}" if _HELD else ""))

    # ── panel (c) input · every rubric at the collapsing arm's endpoint, each read against ITS OWN
    # judged-state distribution: absolute r is not comparable across rubrics (PCT ~0.95, MICI ~0.52).
    _dist = AG.groupby("metric").pearson_r
    SCOPE = AG[AG.model == SAT_LAST.model].copy()
    SCOPE["median_r"] = SCOPE.metric.map(_dist.median())
    SCOPE["p10"] = SCOPE.metric.map(_dist.quantile(0.10))
    SCOPE["p90"] = SCOPE.metric.map(_dist.quantile(0.90))
    SCOPE["n_states"] = SCOPE.metric.map(AG.groupby("metric").size())
    SCOPE["shortfall_vs_median"] = SCOPE.pearson_r - SCOPE.median_r
    SCOPE["rank_in_metric"] = [int((AG.pearson_r[AG.metric == m] < v).sum()) + 1
                               for m, v in zip(SCOPE.metric, SCOPE.pearson_r)]
    SCOPE = SCOPE.sort_values("shortfall_vs_median").reset_index(drop=True)

    # ── the figure ────────────────────────────────────────────────────────────
    # Explicit gridspec margins rather than tight_layout: a manually built GridSpec makes
    # tight_layout warn ("Axes not compatible ... results might be incorrect"), and save_fig's
    # bbox_inches="tight" already trims the outer whitespace.
    fig = plt.figure(figsize=(13.6, 9.0))
    _gs = fig.add_gridspec(2, 2, height_ratios=[1.0, 0.95], width_ratios=[1.0, 1.0],
                           hspace=0.34, wspace=0.20,
                           left=0.055, right=0.995, top=0.90, bottom=0.065)
    axA = fig.add_subplot(_gs[0, :])
    axB, axC = fig.add_subplot(_gs[1, 0]), fig.add_subplot(_gs[1, 1])

    # (a) the symptom — Q1 cross-grader agreement by iteration, one line per arm
    for arm, a in Q.groupby("arm"):
        a, hot = a.sort_values("iteration"), (arm == SAT_ARM)
        axA.plot(a.iteration, a.pearson_r, marker="o", ms=6.5 if hot else 4, lw=2.8 if hot else 1.5,
                 alpha=1.0 if hot else 0.8, color=S.PALETTE.get(arm, "#666666"),
                 zorder=3 if hot else 2,
                 label=eda_analysis.arm_label(arm) + ("  ← collapses" if hot else ""))
    axA.axhline(Q_MEDIAN, ls="--", lw=1.3, color="#444444", zorder=1)
    axA.text(0.015, 0.975,                       # axes fraction: never collides with the lines
             f"dashed = median over all {len(Q)} judged model states = {Q_MEDIAN:.3f}",
             transform=axA.transAxes, fontsize=8.5, color="#444444", va="top", ha="left",
             bbox=dict(boxstyle="round,pad=0.3", fc="#ffffff", ec="#cccccc", alpha=0.9))
    for r in _worst.itertuples():
        axA.annotate(f"{r.pearson_r:.3f}", (r.iteration, r.pearson_r), textcoords="offset points",
                     xytext=(0, -15), ha="center", fontsize=9.5, fontweight="bold", color=SAT_COLOUR)
    axA.set_xticks(sorted(set(Q.iteration)))
    axA.set_xlabel("iteration (0 = base policy)")
    axA.set_ylabel(f"per-conversation r\n({JUDGE_NAME} vs {PRIMARY_NAME})")
    axA.set_title(f"(a) symptom — {SAT_METRIC} cross-grader agreement per model state", loc="left",
                  fontsize=10.5, fontweight="bold")
    axA.legend(frameon=False, fontsize=8.5, ncol=2, loc="lower left")

    # (b) the mechanism — SD under each grader, SAME conversations. Shared y-axis is legitimate
    # here ONLY because SD is a dispersion, not a level.
    _jcolour = {PRIMARY_NAME: "#d55e00", JUDGE_NAME: "#0072b2"}
    _jshort = {PRIMARY_NAME: "primary ", JUDGE_NAME: "held out"}
    for judge, s in SD.groupby("judge"):
        s = s.sort_values("iteration")
        axB.plot(s.iteration, s.sd, marker="o", ms=5.5, lw=2.3, color=_jcolour.get(judge, "#666666"),
                 label=f"{judge} — " + ("TRAINED AGAINST (the reward)" if judge == PRIMARY_NAME
                                             else "held out"))
    # Upper-right is the only quadrant both SD lines leave empty (primary starts high on the left,
    # both end low on the right). The arithmetic is shown, per the composite-number house rule.
    # ⚠ An endpoint-to-endpoint variance ratio is only as stable as its anchor, so each one is
    # printed WITH its trend test. The held-out series' iteration-0 value is its minimum, which is
    # why its ratio looks large (1.41x) while its trend is null - a bare ratio here would invite
    # exactly the "the held-out grader gained spread" reading the trend refutes.
    axB.text(0.985, 0.80,
             f"variance, iter {_i0} → {_iN}   (+ trend over all iters)\n" + "\n".join(
                 f"{_jshort.get(j, j)}:  {_piv.loc[_iN, j]:.3f}² / {_piv.loc[_i0, j]:.3f}²"
                 f" = {VAR_RATIO[j]:.3f}×   ρ={SD_TREND[j]['rho']:+.2f} p={SD_TREND[j]['p']:.3f}"
                 f"  → {SD_TREND[j]['verdict']}"
                 for j in _piv.columns)
             + f"\n(re-anchored to iter {_i0 + 1}: "
             + ", ".join(f"{_jshort.get(j, j).strip()} {SD_TREND[j]['var_ratio_vs_iter1']:.3f}×"
                         for j in _piv.columns) + ")",
             transform=axB.transAxes, fontsize=7.2, va="top", ha="right", family="monospace",
             bbox=dict(boxstyle="round,pad=0.4", fc="#ffffff", ec="#bbbbbb", alpha=0.93))
    # The whole claim is in the two ENDPOINTS (the lines happen to track each other mid-run), so
    # label them rather than leaving the reader to read four values off the axis.
    for judge, s in SD.groupby("judge"):
        s = s.sort_values("iteration")
        _c = _jcolour.get(judge, "#666666")
        _lo, _hi = s.iloc[0], s.iloc[-1]
        axB.annotate(f"{_lo.sd:.3f}", (_lo.iteration, _lo.sd), textcoords="offset points",
                     xytext=(5, 8 if judge == PRIMARY_NAME else -14), ha="left", fontsize=8.5,
                     fontweight="bold", color=_c)
        axB.annotate(f"{_hi.sd:.3f}", (_hi.iteration, _hi.sd), textcoords="offset points",
                     xytext=(7, -2), ha="left", fontsize=8.5, fontweight="bold", color=_c)
    axB.set_xlim(-0.7, int(SD.iteration.max()) + 1.7)
    axB.set_xticks(sorted(set(SD.iteration)))
    axB.set_xlabel("iteration (0 = base policy)")
    axB.set_ylabel(f"SD of per-conversation {SAT_METRIC}\n(each grader in its OWN units)")
    axB.set_title(f"(b) mechanism — {eda_analysis.arm_label(SAT_ARM)}: one ruler saturates, the "
                  f"other does not", loc="left", fontsize=10.5, fontweight="bold")
    axB.legend(frameon=False, fontsize=8.2, loc="upper right")

    # (c) the scope — every rubric at the endpoint state vs its OWN judged-state distribution
    for i, r in enumerate(SCOPE.itertuples()):
        axC.plot([r.p10, r.p90], [i, i], lw=7, color="#dedede", solid_capstyle="butt", zorder=1)
        axC.plot([r.median_r], [i], marker="|", ms=13, mew=2.0, color="#7a7a7a", lw=0, zorder=2)
        axC.plot([r.pearson_r], [i], marker="o", ms=8, color=SAT_COLOUR, zorder=3)
        axC.text(1.045, i, f"{r.pearson_r:5.3f} {r.shortfall_vs_median:+6.3f} "
                           f"{r.rank_in_metric:>2d}/{int(r.n_states)}",
                 va="center", ha="left", fontsize=7.8, family="monospace")
    axC.text(1.045, -0.72, "    r   Δ median   rank", va="center", ha="left", fontsize=7.4,
             family="monospace", color="#555555")
    axC.set_yticks(range(len(SCOPE)))
    axC.set_yticklabels(SCOPE.metric)
    axC.set_ylim(len(SCOPE) - 0.4, -1.1)
    axC.set_xlim(max(0.0, float(SCOPE.p10.min()) - 0.06), 1.46)
    axC.set_xticks([0.2, 0.4, 0.6, 0.8, 1.0])
    axC.set_xlabel(f"per-conversation r at {SAT_LAST.model}")
    axC.set_title(f"(c) scope — each rubric against its own judged-state spread", loc="left",
                  fontsize=10.5, fontweight="bold")
    axC.legend(handles=[
        Line2D([0], [0], color="#dedede", lw=7,
               label=f"10th–90th pct over the {int(SCOPE.n_states.iloc[0])} judged states"),
        Line2D([0], [0], color="#7a7a7a", marker="|", ms=11, mew=2, lw=0, label="that rubric's median"),
        Line2D([0], [0], color=SAT_COLOUR, marker="o", ms=7, lw=0, label=SAT_LAST.model)],
        frameon=False, fontsize=7.4, loc="lower left")

    fig.suptitle(f"[EVAL] Judge saturation — {SAT_METRIC} cross-grader agreement collapses over "
                 f"{eda_analysis.arm_label(SAT_ARM)}'s late iterations, because {SD_STORY}",
                 y=0.975, fontweight="bold", fontsize=11.5)

    # ── the numbers behind all three panels, as one tidy table ────────────────
    _rows = []
    _cross = f"cross-judge ({JUDGE_NAME} vs {PRIMARY_NAME})"
    for r in Q.itertuples():
        _rows.append(dict(panel="a", quantity="cross_judge_pearson_r", metric=SAT_METRIC, arm=r.arm,
                          iteration=int(r.iteration), model=r.model, judge=_cross,
                          value=float(r.pearson_r), n_obs=int(r.n)))
    _rows.append(dict(panel="a", quantity="cross_judge_pearson_r_median_over_judged_states",
                      metric=SAT_METRIC, arm="(all arms)", iteration=np.nan,
                      model=f"{len(Q)} judged states", judge=_cross, value=Q_MEDIAN,
                      n_obs=int(Q.n.sum())))
    for r in SD.itertuples():
        _rows.append(dict(panel="b", quantity="sd_of_per_conversation_score", metric=SAT_METRIC,
                          arm=SAT_ARM, iteration=int(r.iteration), model=r.model, judge=r.judge,
                          value=float(r.sd), n_obs=int(r.n_obs)))
    for j, v in VAR_RATIO.items():
        _rows.append(dict(panel="b", quantity=f"variance_ratio_iter{_i0}_to_iter{_iN}",
                          metric=SAT_METRIC, arm=SAT_ARM, iteration=np.nan,
                          model=f"{SAT_ARM} iter {_i0} -> {_iN}", judge=j, value=float(v),
                          n_obs=int(SD.n_obs[SD.judge == j].sum())))
    for r in SCOPE.itertuples():
        _rows.append(dict(panel="c", quantity="cross_judge_pearson_r", metric=r.metric, arm=SAT_ARM,
                          iteration=int(r.iteration), model=r.model, judge=_cross,
                          value=float(r.pearson_r), n_obs=int(r.n)))
        _rows.append(dict(panel="c", quantity="cross_judge_pearson_r_median_over_judged_states",
                          metric=r.metric, arm="(all arms)", iteration=np.nan,
                          model=f"{int(r.n_states)} judged states", judge=_cross,
                          value=float(r.median_r), n_obs=int(AG.n[AG.metric == r.metric].sum())))
    SATD = pd.DataFrame(_rows)
    SATD["iteration"] = SATD["iteration"].astype("Int64")
    SATD["value"] = SATD["value"].round(4)
    display(SATD.head(20))

    # The arm-level counterweight, read off §2's ladder rather than asserted.
    _pool = SIGN[SIGN.subset == "all contrasts"] if "SIGN" in globals() and not SIGN.empty else None
    SIGN_TXT = (f"{float(_pool.pct_same_sign.iloc[0]):.1f}% of {int(_pool.n_contrasts.iloc[0])} "
                f"arm-level contrasts keep their sign (multijudge_sign_preservation)"
                if _pool is not None and not _pool.empty
                else "the arm-level ladder in multijudge_sign_preservation stays high")

    exports.save_table(SATD, "judge_saturation_data",
                       caption=f"Every number behind judge_saturation, one row per panel element. "
                               f"panel a = {SAT_METRIC} per-conversation Pearson r between {JUDGE_NAME} "
                               f"(held out) and the primary oracle ({PRIMARY_NAME}, which WAS the "
                               f"training reward) for each judged model state, plus the median over "
                               f"all {len(Q)} of them; panel b = the SD of per-conversation "
                               f"{SAT_METRIC} for {SAT_ARM} under each grader separately (a SPREAD in "
                               f"that grader's own units - the two levels are never compared) and the "
                               f"iter {_i0} -> {_iN} variance ratio; panel c = the same agreement at "
                               f"{SAT_LAST.model} for every rubric next to that rubric's own "
                               f"judged-state median. n_obs is conversations behind the value.")
    exports.save_fig(fig, "judge_saturation",
                     caption=f"Judge saturation on {SAT_METRIC}. (a) Per-conversation Pearson r "
                             f"between {JUDGE_NAME} (held out) and the primary oracle ({PRIMARY_NAME}, "
                             f"which WAS the training reward), by iteration, one line per arm; dashed "
                             f"= the median over all {len(Q)} judged model states ({Q_MEDIAN:.3f}). "
                             f"{eda_analysis.arm_label(SAT_ARM)}'s late iterations hold the two "
                             f"lowest-agreeing states in the whole grid "
                             + ", ".join(f"{r.model} r={r.pearson_r:.3f}" for r in _worst.itertuples())
                             + f". (b) The mechanism, over the SAME conversations: {SAT_METRIC} "
                             f"standard deviation by iteration for {eda_analysis.arm_label(SAT_ARM)} "
                             f"under each grader. Both lines share one y-axis because SD is a SPREAD "
                             f"in each grader's own units - it is DISPERSION, not score level, that is "
                             f"being compared (levels differ by 1.2-1.7 points, are model-dependent, "
                             f"and are never compared across graders). The trained-against grader's "
                             f"variance falls to {VAR_RATIO[PRIMARY_NAME]:.3f}x of its base value "
                             f"while the held-out grader's rises to {VAR_RATIO[JUDGE_NAME]:.3f}x, so "
                             f"the conversations did not become identical (that would shrink both) - "
                             f"one ruler saturated. (c) The same agreement at {SAT_LAST.model} across "
                             f"all rubrics, each against ITS OWN judged-state distribution (band = "
                             f"10th-90th percentile, tick = median), since baseline agreement is not "
                             f"comparable across rubrics. ARM-LEVEL vs PER-CONVERSATION: every panel "
                             f"here is per-conversation. The arm-level sign-preservation statistic "
                             f"elsewhere in this family stays high - {SIGN_TXT} - and this figure "
                             f"neither overturns it nor is rescued by it: a high arm-level rate does "
                             f"NOT license a per-conversation claim about these states. {CEIL_TXT} "
                             f"{SUPPORT}Numbers: judge_saturation_data.")
    plt.show()


## 3b · GRPO-only companions — recomputed, not re-scoped  `[EVAL]`

**Purpose.** The GRPO-scoped write-up (`papers/archive/2026_grpo_lookahead_mi (archived 2026-08-27; artifacts kept)`) quotes §2's sign-preservation ladder and §3's agreement medians/ranks — but those are full-grid statistics (all four arms). A paper scoped to the two GRPO arms cannot cite a 44-state median as "the experiment"; the honest move is to **recompute every such number on the 22 GRPO states** and let the write-up quote the recomputed artifact. This cell re-cuts frames §2/§3 already built (`PAIRS`, `AG`, `Q`, `SD`) — no loading, no scoring — into `multijudge_sign_preservation_grpo`, `judge_saturation_grpo` (+ `judge_saturation_grpo_data`, whose panel-c rows carry the per-rubric GRPO-only medians and ranks). The full-grid artifacts above are unchanged and stay canonical for the thesis and the method-contrast paper.


In [ ]:
# ── GRPO-only companions of §2 + §3 — the GRPO-scoped paper's cut ───────────────────────────
# The full-grid artifacts above stay canonical. A write-up scoped to the two GRPO arms cannot
# quote a full-grid median or the full-grid sign-preservation ladder — those numbers must be
# RECOMPUTED on the GRPO states, not re-scoped in prose. Everything here is a re-cut of frames
# already built in this notebook (AG, Q, SD, PAIRS, SIGN); nothing is loaded from disk or scored.
if not ("Q" in globals() and "SD" in globals() and "PAIRS" in globals()):
    print("GRPO-only companions skipped (multi-judge / saturation frames absent).")
else:
    from matplotlib.lines import Line2D

    AG_G = AG[AG.arm.str.startswith("GRPO")]
    QG = Q[Q.arm.str.startswith("GRPO")].sort_values(["arm", "iteration"])
    QG_MEDIAN = float(QG.pearson_r.median())
    _worst_g = QG.nsmallest(2, "pearson_r")
    print(f"[grpo] {SAT_METRIC}: median r = {QG_MEDIAN:.3f} over {len(QG)} GRPO states; two lowest = "
          + ", ".join(f"{r.model} {r.pearson_r:.3f}" for r in _worst_g.itertuples()))

    # ── sign preservation on GRPO-x-GRPO pairs only ───────────────────────────
    PAIRS_G = PAIRS[PAIRS.model_a.str.startswith("GRPOExp3") & PAIRS.model_b.str.startswith("GRPOExp3")]
    _n_states_g = int(pd.concat([PAIRS_G.model_a, PAIRS_G.model_b]).nunique())
    SIGN_G = rel.sign_preservation(PAIRS_G)
    print(f"[grpo] sign preservation over {len(PAIRS_G)} GRPO-x-GRPO contrasts ({_n_states_g} states):")
    display(SIGN_G)
    exports.save_table(SIGN_G, "multijudge_sign_preservation_grpo", caption=(
        f"The sign-preservation ladder restricted to contrasts BETWEEN GRPO states ({len(PAIRS_G)} of the "
        f"{len(PAIRS)} pairwise model-state x metric contrasts; {_n_states_g} states, 8 metrics x C({_n_states_g},2) "
        f"pairs). Companion of multijudge_sign_preservation (all four arms), recomputed for the GRPO-scoped "
        "write-up: same frame, same thresholds, same reading — quote the row at the effect size you are claiming; "
        "the pooled row is dragged down by contrasts too small to claim in the first place."))

    # ── scope: every rubric at the collapsing endpoint vs its own GRPO-only distribution ──────
    _dist_g = AG_G.groupby("metric").pearson_r
    SCOPE_G = AG_G[AG_G.model == SAT_LAST.model].copy()
    SCOPE_G["median_r"] = SCOPE_G.metric.map(_dist_g.median())
    SCOPE_G["p10"] = SCOPE_G.metric.map(_dist_g.quantile(0.10))
    SCOPE_G["p90"] = SCOPE_G.metric.map(_dist_g.quantile(0.90))
    SCOPE_G["n_states"] = SCOPE_G.metric.map(AG_G.groupby("metric").size())
    SCOPE_G["shortfall_vs_median"] = SCOPE_G.pearson_r - SCOPE_G.median_r
    SCOPE_G["rank_in_metric"] = [int((AG_G.pearson_r[AG_G.metric == m] < v).sum()) + 1
                                 for m, v in zip(SCOPE_G.metric, SCOPE_G.pearson_r)]
    SCOPE_G = SCOPE_G.sort_values("shortfall_vs_median").reset_index(drop=True)
    print(f"[grpo] scope at {SAT_LAST.model}, each rubric vs its own {int(SCOPE_G.n_states.iloc[0])}-state "
          "GRPO distribution:")
    display(SCOPE_G[["metric", "pearson_r", "median_r", "shortfall_vs_median", "rank_in_metric", "n_states"]].round(3))

    # ── the two-panel figure: (a) agreement GRPO-only; (b) the SD mechanism, unchanged ────────
    fig, (axA, axB) = plt.subplots(2, 1, figsize=(7.4, 7.6))
    for arm, a in QG.groupby("arm"):
        a, hot = a.sort_values("iteration"), (arm == SAT_ARM)
        axA.plot(a.iteration, a.pearson_r, marker="o", ms=6.5 if hot else 4.5, lw=2.6 if hot else 1.6,
                 alpha=1.0 if hot else 0.85, color=S.PALETTE.get(arm, "#666666"),
                 zorder=3 if hot else 2,
                 label=eda_analysis.arm_label(arm) + ("  ← collapses" if hot else ""))
    axA.axhline(QG_MEDIAN, ls="--", lw=1.2, color="#444444", zorder=1)
    axA.text(0.015, 0.06, f"dashed = median over the {len(QG)} GRPO model states = {QG_MEDIAN:.3f}",
             transform=axA.transAxes, fontsize=8.5, color="#444444", va="bottom", ha="left",
             bbox=dict(boxstyle="round,pad=0.3", fc="#ffffff", ec="#cccccc", alpha=0.9))
    for r in _worst_g.itertuples():
        axA.annotate(f"{r.pearson_r:.3f}", (r.iteration, r.pearson_r), textcoords="offset points",
                     xytext=(0, -15), ha="center", fontsize=9.5, fontweight="bold", color=SAT_COLOUR)
    axA.set_xticks(sorted(set(QG.iteration)))
    axA.set_xlabel("iteration (0 = base policy)")
    axA.set_ylabel(f"per-conversation r\n({JUDGE_NAME} vs {PRIMARY_NAME})")
    axA.set_title(f"(a) symptom — {SAT_METRIC} cross-grader agreement per GRPO model state", loc="left",
                  fontsize=10.5, fontweight="bold")
    axA.legend(frameon=False, fontsize=8.5, loc="lower left", bbox_to_anchor=(0.0, 0.16))
    axA.grid(True, alpha=0.25)

    _jcolour = {PRIMARY_NAME: "#d55e00", JUDGE_NAME: "#0072b2"}
    _jshort = {PRIMARY_NAME: "primary ", JUDGE_NAME: "held out"}
    for judge, s in SD.groupby("judge"):
        s = s.sort_values("iteration")
        axB.plot(s.iteration, s.sd, marker="o", ms=5, lw=2.0, color=_jcolour.get(judge, "#666666"),
                 label=f"{judge} — " + ("TRAINED AGAINST (the reward)" if judge == PRIMARY_NAME
                                        else "held out"))
        _c = _jcolour.get(judge, "#666666")
        _lo, _hi = s.iloc[0], s.iloc[-1]
        axB.annotate(f"{_lo.sd:.3f}", (_lo.iteration, _lo.sd), textcoords="offset points",
                     xytext=(5, 8 if judge == PRIMARY_NAME else -14), ha="left", fontsize=8.5,
                     fontweight="bold", color=_c)
        axB.annotate(f"{_hi.sd:.3f}", (_hi.iteration, _hi.sd), textcoords="offset points",
                     xytext=(7, -2), ha="left", fontsize=8.5, fontweight="bold", color=_c)
    axB.text(0.985, 0.80,
             f"variance, iter {_i0} → {_iN}   (+ trend over all iters)\n" + "\n".join(
                 f"{_jshort.get(j, j)}:  {_piv.loc[_iN, j]:.3f}² / {_piv.loc[_i0, j]:.3f}²"
                 f" = {VAR_RATIO[j]:.3f}×   ρ={SD_TREND[j]['rho']:+.2f} p={SD_TREND[j]['p']:.3f}"
                 f"  → {SD_TREND[j]['verdict']}"
                 for j in _piv.columns)
             + f"\n(re-anchored to iter {_i0 + 1}: "
             + ", ".join(f"{_jshort.get(j, j).strip()} {SD_TREND[j]['var_ratio_vs_iter1']:.3f}×"
                         for j in _piv.columns) + ")",
             transform=axB.transAxes, fontsize=7.2, va="top", ha="right", family="monospace",
             bbox=dict(boxstyle="round,pad=0.4", fc="#ffffff", ec="#bbbbbb", alpha=0.93))
    axB.set_xlim(-0.7, int(SD.iteration.max()) + 1.7)
    axB.set_xticks(sorted(set(SD.iteration)))
    axB.set_xlabel("iteration (0 = base policy)")
    axB.set_ylabel(f"SD of per-conversation {SAT_METRIC}\n(each grader in its OWN units)")
    axB.set_title(f"(b) mechanism — {eda_analysis.arm_label(SAT_ARM)}: one ruler saturates, the "
                  f"other does not", loc="left", fontsize=10.5, fontweight="bold")
    axB.legend(frameon=False, fontsize=8.2, loc="upper right", bbox_to_anchor=(1.0, 0.62))
    axB.grid(True, alpha=0.25)
    fig.suptitle(f"[EVAL] Judge saturation, GRPO arms only — the GRPO-scoped companion of "
                 f"judge_saturation", y=0.985, fontweight="bold", fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, 0.965))

    # ── the numbers behind both panels + the scope re-cut, one tidy table ─────
    _rows_g = []
    _cross = f"cross-judge ({JUDGE_NAME} vs {PRIMARY_NAME})"
    for r in QG.itertuples():
        _rows_g.append(dict(panel="a", quantity="cross_judge_pearson_r", metric=SAT_METRIC, arm=r.arm,
                            iteration=int(r.iteration), model=r.model, judge=_cross,
                            value=float(r.pearson_r), n_obs=int(r.n)))
    _rows_g.append(dict(panel="a", quantity="cross_judge_pearson_r_median_over_grpo_states",
                        metric=SAT_METRIC, arm="(GRPO arms)", iteration=np.nan,
                        model=f"{len(QG)} GRPO states", judge=_cross, value=QG_MEDIAN,
                        n_obs=int(QG.n.sum())))
    for r in SD.itertuples():
        _rows_g.append(dict(panel="b", quantity="sd_of_per_conversation_score", metric=SAT_METRIC,
                            arm=SAT_ARM, iteration=int(r.iteration), model=r.model, judge=r.judge,
                            value=float(r.sd), n_obs=int(r.n_obs)))
    for r in SCOPE_G.itertuples():
        _rows_g.append(dict(panel="c", quantity="cross_judge_pearson_r", metric=r.metric, arm=SAT_ARM,
                            iteration=int(r.iteration), model=r.model, judge=_cross,
                            value=float(r.pearson_r), n_obs=int(r.n)))
        _rows_g.append(dict(panel="c", quantity="cross_judge_pearson_r_median_over_grpo_states",
                            metric=r.metric, arm="(GRPO arms)", iteration=np.nan,
                            model=f"{int(r.n_states)} GRPO states", judge=_cross,
                            value=float(r.median_r), n_obs=int(AG_G.n[AG_G.metric == r.metric].sum())))
        _rows_g.append(dict(panel="c", quantity="rank_in_metric_among_grpo_states",
                            metric=r.metric, arm=SAT_ARM, iteration=int(r.iteration), model=r.model,
                            judge=_cross, value=float(r.rank_in_metric), n_obs=int(r.n_states)))
    SATG = pd.DataFrame(_rows_g)
    SATG["iteration"] = SATG["iteration"].astype("Int64")
    SATG["value"] = SATG["value"].round(4)

    exports.save_table(SATG, "judge_saturation_grpo_data", caption=(
        f"Every number behind judge_saturation_grpo, one row per panel element — the GRPO-only re-cut of "
        f"judge_saturation_data. panel a = {SAT_METRIC} per-conversation Pearson r between {JUDGE_NAME} (held out) "
        f"and the primary oracle ({PRIMARY_NAME}, which WAS the training reward) for each GRPO model state, plus "
        f"the median over the {len(QG)} of them (NOT the full-grid median — quote this one in any GRPO-scoped "
        f"write-up); panel b = the SD rows, identical to judge_saturation_data's (that panel was already "
        f"{SAT_ARM}-only); panel c = the same agreement at {SAT_LAST.model} for every rubric next to that rubric's "
        f"own GRPO-only median and its rank among the GRPO states (1 = worst-agreeing GRPO state on that rubric). "
        f"n_obs is conversations behind the value (n_states for rank rows)."))
    exports.save_fig(fig, "judge_saturation_grpo", caption=(
        f"**Judge saturation, GRPO arms only** — the GRPO-scoped companion of judge_saturation (which carries all "
        f"four arms and stays canonical). (a) Per-conversation Pearson r between {JUDGE_NAME} (held out) and the "
        f"primary oracle ({PRIMARY_NAME}, which WAS the training reward) on {SAT_METRIC}, by iteration, the two "
        f"GRPO arms only; dashed = the median over the {len(QG)} GRPO model states ({QG_MEDIAN:.3f}). "
        f"{eda_analysis.arm_label(SAT_ARM)}'s late iterations are the lowest-agreeing GRPO states: "
        + ", ".join(f"{r.model} r={r.pearson_r:.3f}" for r in _worst_g.itertuples())
        + f". (b) The mechanism, over the SAME conversations — identical to judge_saturation's panel (b), which "
        f"was already {SAT_ARM}-only: {SAT_METRIC} SD by iteration under each grader; one y-axis because SD is a "
        f"SPREAD in each grader's own units (levels differ by 1.2-1.7 points and are never compared across "
        f"graders). The trained-against grader's variance falls to {VAR_RATIO[PRIMARY_NAME]:.3f}x of its base "
        f"value with a monotone trend (ρ={SD_TREND[PRIMARY_NAME]['rho']:+.2f}, p={SD_TREND[PRIMARY_NAME]['p']:.3f}) "
        f"while the held-out grader's trend is null — one ruler saturated; had the conversations become alike, "
        f"both spreads would have shrunk. Scope-by-rubric lives in the table (panel c rows of "
        f"judge_saturation_grpo_data). {CEIL_TXT} Numbers: judge_saturation_grpo_data."))
    plt.show()


## 4 · How to read this notebook
- **ICC (§1)** is how much of a per-conversation score is signal rather than re-scoring noise. Read it against Koo & Li (2016): ≥0.75 good, ≥0.90 excellent. It bounds everything downstream — an arm difference smaller than the grader's own noise is not a difference.
- **Cross-judge `r` must be compared to the `ceiling`, never to 1.0.** The ceiling is `sqrt(ICC_primary × ICC_judge)`: two imperfect raters cannot correlate perfectly even when measuring the same thing. Both terms have been measured since 2026-07-28; `ceiling_basis` records whether a cell used measured values or fell back to the old `ICC_judge == ICC_primary` assumption.
- **`same_sign` is the load-bearing number**, not the correlation. A large level `bias` between judges is expected and harmless — the thesis reports contrasts, which cancel it. What would hurt is an arm *ordering* that depends on who grades.
- **Never average the two judges' raw scores.** The primary oracle *was the training reward*; the second judge never touched training. That is optimization-target vs held-out-test, not two interchangeable raters — `reliability.py` enforces this and only ever combines contrasts or standardized quantities.
- **`arm × judge` (§2) is the only variance component that can invalidate a claim.** A large `judge` term is a level shift; a large interaction means the ranking itself moves with the grader. Read `dependability_k1` as "how far can I trust an arm ranking taken off ONE judge".
- **Gain retention (§2) is the reward-hacking test**: `Δ(held-out) / Δ(trained-against)`. ~1.0 = a real behaviour change both graders see; ~0 = a gain that existed only in the optimized grader.
- **Concordance (§2) is per conversation PAIR, not per arm** — arm means over 96 conversations resolve ~10× better, so do not read a bin height as confidence in an arm-level claim.
- **Judge saturation (§3) is per conversation and describes the INSTRUMENT, not the policy.** A cross-grader correlation can fall for two very different reasons — the conversations became indistinguishable (both graders' spreads shrink) or one grader ran out of range (the spreads move in opposite directions) — and only the second is a measurement failure. §3 separates them by plotting each grader's SD on the same conversations. It is orthogonal to §2's arm-level ladder: neither one licenses a conclusion at the other's level of analysis.
- _(The measured values are narrated in `results/measurement/SUMMARY.md` and the caveats they imply in `results/LIMITATIONS.md` §1–§3. This notebook is where they are computed.)_


In [ ]:
exports.prune_orphan_captions(); print("index ->", exports.build_index())